# Chapter 6 - Building Your First Deep Neural Network: Introduction to Backpropagation

## Summary
Learn the **streetlight problem**: map traffic-light patterns (walk/stop) to a target. A single-layer net cannot solve it when there is *no direct correlation* between any one input and output.
- **Correlation:** a network finds correlations; an input with correlation to the output gets its weight pushed up.
- **Hidden layers + nonlinearity (ReLU):** create *intermediate correlation*. Without non-linearity, stacked layers collapse into one linear layer.
- **Backpropagation:** "weighted average of the deltas" - propagate the output delta back to earlier layers (`layer_1_delta = layer_2_delta . W2^T * relu2deriv(layer_1)`), then update every weight.
- Full algorithm: forward pass, compute deltas backwards, update weights with `layer.T.dot(delta)`.

## Theory
- **ReLU:** `relu(x) = max(0, x)`; derivative 1 if x>0 else 0. Lets some hidden neurons switch off.
- **Chain rule:** the error's effect on a hidden weight = (effect on output) x (effect of hidden unit on output) x (effect of weight on hidden unit).
- **Backprop:** efficiently applies the chain rule layer by layer, reusing deltas.
- **Why it works:** hidden layers give the model *conditional* correlations (e.g. 'neuron fires only if left AND right light off').

In [1]:
import numpy as np
np.random.seed(1)

streetlights = np.array([[1,0,1],[0,1,1],[0,0,1],[1,1,1]])
walk_vs_stop = np.array([[1,1,0,0]]).T

# Single layer network learns the 'dominant' correlation
weights = np.random.random((3,1)) * 0.2 - 0.1
for it in range(40):
    err = 0
    for i in range(len(streetlights)):
        layer = streetlights[i:i+1]
        pred = layer.dot(weights)
        delta = pred - walk_vs_stop[i:i+1]
        err += (delta ** 2).sum()
        weights -= 0.1 * layer.T.dot(delta)
    if it % 10 == 9: print("Error:", round(err, 5))
print(weights.round(3))

Error: 1.29653
Error: 1.25806
Error: 1.24457
Error: 1.23916
[[ 0.013]
 [-0.042]
 [ 0.427]]


In [2]:
# Full 3-layer network with ReLU + backprop (streetlight problem)
np.random.seed(1)
def relu(x): return (x > 0) * x
def relu2deriv(o): return o > 0

alpha, hidden_size = 0.2, 4
w01 = 2 * np.random.random((3, hidden_size)) - 1
w12 = 2 * np.random.random((hidden_size, 1)) - 1

for it in range(60):
    err = 0
    for i in range(len(streetlights)):
        l0 = streetlights[i:i+1]
        l1 = relu(l0.dot(w01))
        l2 = l1.dot(w12)
        err += ((l2 - walk_vs_stop[i:i+1]) ** 2).sum()
        l2_delta = l2 - walk_vs_stop[i:i+1]
        l1_delta = l2_delta.dot(w12.T) * relu2deriv(l1)
        w12 -= alpha * l1.T.dot(l2_delta)
        w01 -= alpha * l0.T.dot(l1_delta)
    if it % 10 == 9: print("Error:", round(err, 6))

Error: 0.634231
Error: 0.358384
Error: 0.083018
Error: 0.006467
Error: 0.000329
Error: 1.5e-05


In [3]:
# Check final predictions
for l0 in streetlights:
    print(l0, "->", relu(l0.dot(w01)).dot(w12).round(3))

[1 0 1] -> [1.]
[0 1 1] -> [0.999]
[0 0 1] -> [0.003]
[1 1 1] -> [0.]


## Takeaways
Non-linear hidden layers + backpropagation allow a network to learn patterns invisible to single-layer nets. This is the foundation of every deep network.